# 要掌握如何给模型喂多个输入

In [1]:
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F

print(sys.version_info)
for module in mpl, np, pd, sklearn, torch:
    print(module.__name__, module.__version__)
    
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(device)


sys.version_info(major=3, minor=12, micro=3, releaselevel='final', serial=0)
matplotlib 3.10.1
numpy 2.2.4
pandas 2.2.3
sklearn 1.6.1
torch 2.7.0+cpu
cpu


In [2]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
print(housing.DESCR)
print(housing.data.shape)
print(housing.target.shape)

D:\Python312\Lib\site-packages\sklearn\datasets\_base.py:1519: UserWarning: Retry downloading from url: https://ndownloader.figshare.com/files/5976036
  warnings.warn(f"Retry downloading from url: {remote.url}")


URLError: <urlopen error [WinError 10061] 由于目标计算机积极拒绝，无法连接。>

In [ ]:
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader


scaler = StandardScaler()
scaler.fit(x_train)

In [ ]:
from sklearn.model_selection import train_test_split

#拆分训练集和测试集，random_state是随机种子,同样的随机数种子，是为了得到同样的随机值
x_train_all, x_test, y_train_all, y_test = train_test_split(
    housing.data, housing.target, random_state = 7)
x_train, x_valid, y_train, y_valid = train_test_split(
    x_train_all, y_train_all, random_state = 11)
# 训练集
print(x_train.shape, y_train.shape)
# 验证集
print(x_valid.shape, y_valid.shape)
# 测试集
print(x_test.shape, y_test.shape)

dataset_maps = {
    "train": [x_train, y_train],
    "valid": [x_valid, y_valid],
    "test": [x_test, y_test],
}


### 构建数据集

这里我们构建多输入的数据集，注意到数据集介绍里对每个特征定义如下：

> **Attribute Information**:
> - MedInc        median income in block group
> - HouseAge      median house age in block group
> - AveRooms      average number of rooms per household
> - AveBedrms     average number of bedrooms per household
> - Population    block group population
> - AveOccup      average number of household members
> - Latitude      block group latitude
> - Longitude     block group longitude

我们认为最后两维作为位置信息要单独处理，故制作数据集如下

In [ ]:
from torch.utils.data import Dataset

class HousingDataset(Dataset):
    def __init__(self, mode='train'):
        self.x, self.y = dataset_maps[mode]
        self.x = torch.from_numpy(scaler.transform(self.x)).float()
        self.y = torch.from_numpy(self.y).float().reshape(-1, 1)
            
    def __len__(self):
        return len(self.x)
    
    # 这里十分重要，如何返回多个输入
    def __getitem__(self, idx):
        return (self.x[idx],self.x[idx][-2:]), self.y[idx] #返回的是一个元组，元组里面是两个元素，第一个元素是一个元组，第二个元素是一个数。self.x[idx][-2:]代表取最后两个元素
    
    
train_ds = HousingDataset("train")
valid_ds = HousingDataset("valid")
test_ds = HousingDataset("test")

In [ ]:
from torch.utils.data import DataLoader


batch_size = 8
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

# 定义模型

In [ ]:
#回归模型我们只需要1个数

class WideDeep(nn.Module):
    def __init__(self, input_dim=[8, 2]): # 多个输入的维度
        super().__init__()
        self.deep = nn.Sequential(
            nn.Linear(input_dim[1], 30),
            nn.ReLU(),
            nn.Linear(30, 30),
            nn.ReLU()
            )
        # pytorch 需要自行计算输出输出维度
        self.output_layer = nn.Linear(30 + input_dim[0], 1)
        
        # 初始化权重
        self.init_weights()
        
    def init_weights(self):
        """使用 xavier 均匀分布来初始化全连接层的权重 W"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
        
    def forward(self, x_wide, x_deep):
        # x_deep.shape [batch size, 2]
        # x_wide.shape [batch size, 8]
        deep_output = self.deep(x_deep)
        # concat [batch size, 30] with [batch size 8]
        concat = torch.cat([x_wide, deep_output], dim=1)
        logits = self.output_layer(concat)
        # logits.shape [batch size, 1]
        return logits